# Saber ffberm_v2 Sweep — GSM8K (LIMIT=150)

`ffberm_v2`: rewarm 时的 BERM 使用**扩窗前的 last_conf** 作为基准（而非零向量），
使得 delta = new_conf - old_conf 有意义。新扩出的 block 位置用 x0_p 填充（无历史可比，不会被退回）。

日常 refinement step 的 BERM 和 `current_block` 一样，只在最新 watching block 内操作。

12 个任务 + 1 个 anchor。

## 1. 环境设置

In [ ]:
import os, gc, re, json, datetime, threading, queue, subprocess
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '4,5')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)
os.makedirs('evals_results/saber', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

## 2. 任务配置

In [ ]:
task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 256
steps = 256
limit_samples = 150
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def _task(name, n, mu, berm_scope='window'):
    extra_args = [
        'saber_expand=True',
        'berm_mode=cross_step',
        'saber_global_aadu=True',
        'saber_mtr=0.8',
        f'saber_n={n}',
        f'saber_mu={mu}',
        'block_length=32',
        f'saber_berm_scope={berm_scope}',
    ]
    return {'name': name, 'extra_args': extra_args}


TASK_CONFIGS = [
    # anchor
    _task('ffv2_anchor_n4_mu8', 4, 8, 'window'),

    # ffberm_v2: n sweep (mu=8)
    _task('ffv2_n3_mu8', 3, 8, 'ffberm_v2'),
    _task('ffv2_n4_mu8', 4, 8, 'ffberm_v2'),
    _task('ffv2_n5_mu8', 5, 8, 'ffberm_v2'),
    _task('ffv2_n6_mu8', 6, 8, 'ffberm_v2'),
    _task('ffv2_n8_mu8', 8, 8, 'ffberm_v2'),

    # ffberm_v2: mu sweep (n=4)
    _task('ffv2_n4_mu4', 4, 4, 'ffberm_v2'),
    _task('ffv2_n4_mu6', 4, 6, 'ffberm_v2'),
    _task('ffv2_n4_mu10', 4, 10, 'ffberm_v2'),
    _task('ffv2_n4_mu12', 4, 12, 'ffberm_v2'),

    # ffberm_v2: cross combos
    _task('ffv2_n5_mu12', 5, 12, 'ffberm_v2'),
    _task('ffv2_n6_mu4', 6, 4, 'ffberm_v2'),
    _task('ffv2_n3_mu12', 3, 12, 'ffberm_v2'),
]

GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
print(f'Total tasks: {len(TASK_CONFIGS)} | GPUs: {GPU_POOL} | limit: {limit_samples} | timestamp: {timestamp}')

## 3. 并行启动任务

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_ffberm_v2_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

        common_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            'show_speed=True',
            f'seed={seed}',
        ]
        model_args = ','.join(common_args + cfg['extra_args'])
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} --confirm_run_unsafe_code '
            f'--model llada_dist --model_args {model_args} --output_path {output_dir} '
            f'--log_samples --limit {limit_samples}'
        )

        print(f'[GPU {gpu_id}] START {name}')
        p = subprocess.Popen(
            cmd,
            shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()
        print(f'[GPU {gpu_id}] DONE  {name} rc={rc}')
        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))
        task_queue.task_done()


threads = []
for gid in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print('All finished:', len(all_results), '/', len(TASK_CONFIGS))

## 4. 解析评测结果

In [ ]:
def parse_result(cfg, output_dir, log_file):
    name = cfg['name']

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    return {
        'name': name,
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


rows = []
for cfg, name, log_file, output_dir, rc in all_results:
    rows.append(parse_result(cfg, output_dir, log_file))

df = pd.DataFrame(rows).sort_values(['n', 'mu'])
pd.set_option('display.max_rows', 50)

print(f'\n{"Name":<26} {"n":<4} {"mu":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 84)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<26} {r['n']:<4} {r['mu']:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

## 5. 对比图

In [ ]:
plot_df = df[df['flex_acc'].notna()].copy()
plot_df['label'] = plot_df['name']

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
metrics = [('flex_acc', 'FlexAcc'), ('total_nfe', 'Total NFE'), ('tok_per_sec', 'Tokens/sec')]
colors = ['#E53935' if 'anchor' in n else '#59A14F' for n in plot_df['name']]

for ax, (col, title) in zip(axes, metrics):
    vals = plot_df[col].fillna(0)
    ax.bar(plot_df['label'], vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].tick_params(axis='x', rotation=55, labelsize=8)
plt.tight_layout()
plt.show()

## 6. 从已有结果重新加载（可选）

内核重启后运行此 cell + Section 5。

In [ ]:
import glob, re, json, os
from pathlib import Path
import pandas as pd

task = 'gsm8k'
seed = 42

NAMES = [
    'ffv2_anchor_n4_mu8',
    'ffv2_n3_mu8',
    'ffv2_n4_mu8',
    'ffv2_n5_mu8',
    'ffv2_n6_mu8',
    'ffv2_n8_mu8',
    'ffv2_n4_mu4',
    'ffv2_n4_mu6',
    'ffv2_n4_mu10',
    'ffv2_n4_mu12',
    'ffv2_n5_mu12',
    'ffv2_n6_mu4',
    'ffv2_n3_mu12',
]

latest = sorted(
    glob.glob(f'nlogs/sweep_ffberm_v2_{task}_ffv2_anchor_n4_mu8_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest:
    fname = os.path.basename(latest[0])
    timestamp = fname.replace(f'sweep_ffberm_v2_{task}_ffv2_anchor_n4_mu8_', '').replace('.log', '')
    print(f'Auto-detected timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

parsed_results = []
for name in NAMES:
    lf = f'nlogs/sweep_ffberm_v2_{task}_{name}_{timestamp}.log'
    od = f'evals_results/saber/{task}-{name}-{timestamp}'
    if not os.path.exists(lf):
        print(f'  [MISS] {name}')
        continue

    log_content = Path(lf).read_text(encoding='utf-8', errors='ignore')
    result_json = Path(od) / 'results.json'
    flex_acc = strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')
    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))
    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None: speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None: tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None: time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    parsed_results.append({
        'name': name,
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })
    print(f'  [OK]   {name}')

df = pd.DataFrame(parsed_results).sort_values(['n', 'mu'])
print(f'\nLoaded {len(parsed_results)} results (timestamp={timestamp})')
print(f'\n{"Name":<26} {"n":<4} {"mu":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 84)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<26} {r['n']:<4} {r['mu']:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

print(f'\nRe-run Section 5 to regenerate plots.')